# Word sentiment EEG — onsets and class ERPs

Use a **CPU** runtime. Choose **Runtime → Run all** and allow Drive mounting. The GitHub repository is public; no GitHub token or Colab secret is required.

Uses the prepared agent labels across ZuCo 1.0 **SR, NR and TSR**. Uncertain labels, lexical/context disagreements and alignment problems are excluded automatically. No label review or editing is required.

Extracts the first-fixation onset of each word, −200 to +800 ms EEG epochs, and positive/negative/neutral fixation-related ERP averages. Existing source files are reused. Missing MATLAB files download **only in Colab**, one at a time, and are deleted from Colab after extraction. Completed readers resume from Drive checkpoints.

**Drive organization:** small reports and figures in `Thesis/Results/word_sentiment_eeg/v3_word_erp`; reusable class epochs in `Thesis/CachedArtifacts/word_sentiment_eeg/v3_word_erp`. No duplicate original EEG files are added to Drive. Keeping all accepted 105-channel trial epochs can require tens of GB on Drive; the actual size depends on accepted epochs and compression. CPU runtime is sufficient; the full first run may take hours. Leave it running. Re-run All after a disconnect to reuse completed readers.

The annotations are single-agent judgments, not human consensus. Descriptive FRPs may contain overlapping responses to adjacent fixations. Scientific results are pending until the run finishes.


In [ ]:
# Run all. This cell runs only in Colab; nothing is installed on your Mac.
from google.colab import drive
from pathlib import Path
import os, subprocess, sys
from IPython.display import Markdown, display

drive.mount('/content/drive')
THESIS = Path('/content/drive/MyDrive/Thesis')
REPO = Path('/content/word-sentiment-eeg')
REPO_URL = 'https://github.com/parmisbathayan/word-sentiment-eeg.git'
RUN_TAG = 'v3_word_erp'

# Public repository: clone/update without tokens or credential prompts.
def git(*args):
    result = subprocess.run(
        ['git', '-c', 'credential.helper=', '-c', 'http.extraheader=', *map(str, args)],
        env=dict(os.environ, GIT_TERMINAL_PROMPT='0'), capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError('Could not fetch the public GitHub repository. Check the Colab internet connection and any local checkout edits, then rerun. No GitHub token is required.')
if not REPO.exists():
    git('clone', '--depth', '1', '--branch', 'main', REPO_URL, REPO)
else:
    if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('The Colab code checkout has edits. Save them before updating the checkout.')
    git('-C', REPO, 'pull', '--ff-only', 'origin', 'main')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-colab.txt')], check=True)
print('Code version:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('Public GitHub code loaded without a token. CPU is sufficient.')


In [ ]:
# Extract word onsets, class EEG epochs and ERP averages. No classification training.
command = [sys.executable, '-u', str(REPO / 'run_erp.py'), '--thesis-root', str(THESIS), '--run-tag', RUN_TAG, '--allow-cloud-download']
subprocess.run(command, check=True)


In [ ]:
RUN = THESIS / 'Results/word_sentiment_eeg' / RUN_TAG
if (RUN / 'REPORT.md').exists():
    display(Markdown((RUN / 'REPORT.md').read_text()))
else:
    print('The run is incomplete. See', RUN / 'run.log')
from IPython.display import Image
for task in ['SR', 'NR', 'TSR']:
    figure = RUN / f'plots/{task}_class_erp.png'
    if figure.exists():
        display(Image(filename=str(figure)))
print('Results:', RUN)
print('EEG epochs:', THESIS / 'CachedArtifacts/word_sentiment_eeg' / RUN_TAG)
print('When finished, tell Codex: the word ERP run finished. Codex can inspect the saved report and result tables through Drive.')
